In [0]:
# Databricks notebook source
"""Run data quality checks on the silver PSA population table."""

from __future__ import annotations

import sys

from pyspark.sql import Column, SparkSession
from pyspark.sql.functions import col, count, lit

# COMMAND ----------

SILVER_TABLE = "ahon.silver.psa_population"
MERGE_KEYS = ["geographic_location", "census_year"]
MAX_PERCENT = 100
INFO_CHECKS = {"total_rows"}  # reported only, never counted as a failure

# COMMAND ----------


def report(message: str) -> None:
    """Write a line to stdout, which Databricks shows as cell output."""
    sys.stdout.write(f"{message}\n")


def build_row_checks() -> dict[str, Column]:
    """Return the row-level checks, each true for a failing row.

    The conditions are built inside a function, not at import time, because
    pyspark's col() needs an active Spark session.
    """
    return {
        "null_geographic_location": col("geographic_location").isNull(),
        "null_total_population": col("total_population").isNull(),
        "null_urban_population": col("urban_population").isNull(),
        "null_percent_urban": col("percent_urban").isNull(),
        "negative_total_population": col("total_population") < 0,
        "negative_urban_population": col("urban_population") < 0,
        "percent_urban_out_of_range": (
            (col("percent_urban") < 0) | (col("percent_urban") > MAX_PERCENT)
        ),
        "urban_gt_total_population": col("urban_population") > col("total_population"),
    }


def run_silver_dq_checks(spark: SparkSession) -> dict[str, int]:
    """Run data quality checks on the silver table and print a summary.

    Returns a dict mapping each check name to its failing row count, where 0
    means the check passed. The total row count is included for reference.
    """
    silver = spark.table(SILVER_TABLE)

    results = {"total_rows": silver.count()}
    for name, failing in build_row_checks().items():
        results[name] = silver.filter(failing).count()
    results["duplicate_merge_keys"] = (
        silver.groupBy(*MERGE_KEYS)
        .agg(count(lit(1)).alias("n"))
        .filter(col("n") > 1)
        .count()
    )

    report("-- Silver DQ Checks --")
    for check, value in results.items():
        status = "PASS" if check in INFO_CHECKS or value == 0 else "FAIL"
        report(f"  {check:40s} {value:>10,}  {status}")
    return results


def main() -> None:
    """Run silver DQ checks and print the summary."""
    spark = SparkSession.builder.getOrCreate()
    run_silver_dq_checks(spark)


# COMMAND ----------

if __name__ == "__main__":
    main()